In [ ]:
!pip install bitsandbytes
#Biblioteca para Quantização de modelos

#https://huggingface.co/docs/datasets/index
from datasets import load_dataset
#Excelente biblioteca para lidar com grandes volumes de dados pela sua capacidade de streaming, cache e suporte.
#Baseada em Apache Arrow que é excelente para big data pelo seu formato colunar. Menos RAM, menos cópias e memory mapping (Vai inserindo na RAM à medida que o dataset é usado).
#--------------------------------------------

#https://arxiv.org/pdf/1912.01703
import torch
#Library ABC para Deep Learnig
#--------------------------------------------

#https://huggingface.co/docs/accelerate/index
import accelerate
#Apesar de não usarmos diretamente, quero realçar esta biblioteca que muitas das bibliotecas aqui importadas utilizam por baixo do capô.
#--------------------------------------------

#https://huggingface.co/docs/transformers/index
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    TrainerCallback,
    Trainer
)
#Biblioteca muito famosa na área de IA pela sua facilidade de uso. Mantida pela Hugging Face, a maior comunidade de LLM's.
#--------------------------------------------

#https://huggingface.co/docs/peft/index
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
#Biblioteca para realizar Parameter-Efficiente Fine-Tuning. Mantida também pela Hugging Face.
#--------------------------------------------

import time
import psutil #https://psutil.readthedocs.io/stable/
from pynvml import * #https://developer.nvidia.com/management-library-nvml

import logging #https://docs.python.org/3/library/logging.html
import yaml
import numpy as np # Para Estatísticas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.4 MB/s eta 0:00:00


In [ ]:
with open ("Config.yaml", "r", encoding = "utf-8") as f:
    CONFIG = yaml.safe_load (f)

print (CONFIG)
print (CONFIG.keys())

In [ ]:
logging.basicConfig (filename = "training.log", level = logging.INFO, format = "%(asctime)s | %(message)s", force = True)


<hr>

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

BNB_CONFIG = BitsAndBytesConfig (
      load_in_4bit = True,  #Quantização
      bnb_4bit_quant_type = 'nf4',  #Tipo de Quantização #https://huggingface.co/docs/transformers/v5.13.1/en/quantization/bitsandbytes?bnb=4-bit#normal-float-4-nf4
      bnb_4bit_use_double_quant = True, #Double Quantization #https://huggingface.co/docs/transformers/v5.13.1/en/quantization/bitsandbytes?bnb=4-bit#nested-quantization
      bnb_4bit_compute_dtype = torch.float16 #https://huggingface.co/docs/transformers/v5.13.1/en/quantization/bitsandbytes?bnb=4-bit#compute-data-type #https://bitfern.com/blog/bf16-vs-fp16/#BF16_vs_FP16_The_Key_Differences
      #FP16 - (1, 5, 10) #BF16 - (1, 8, 7) #Ter atenção à GPU, T4 não aguenta BF16. #https://perf.svcfusion.com/ #https://docs.pytorch.org/docs/main/tensor_attributes.html
    )

TOKENIZER = AutoTokenizer.from_pretrained (CONFIG["Modelo"]["nome"])
MODELO = AutoModelForCausalLM.from_pretrained (CONFIG["Modelo"]["nome"], device_map = device, dtype = torch.float16, quantization_config = BNB_CONFIG) #sdpa
#https://huggingface.co/docs/transformers/v5.13.1/en/main_classes/model#transformers.PreTrainedModel.from_pretrained
#device_map = "auto" deixa accelerate a tratar do load dos pesos do modelo. É o recomendado, em vez de to.device
#quantization_config é usado para aplicar quantização ao modelo com BitsAndBytes

#https://huggingface.co/docs/transformers/v5.13.1/en/main_classes/model#transformers.PreTrainedModel.from_pretrained.attn_implementation
#attn_implementation aplica técnicas eficientes de realizar Atenção. Flash Attention seria o ideal mas a GPU T4 não tem suporte por isso usamos Scaled Dot Product Attention Otimizada by Pytorch.

#Preparar o modelo Quantizado para LoRA #https://github.com/huggingface/peft/blob/main/src/peft/utils/other.py
MODELO = prepare_model_for_kbit_training (MODELO)
MODELO.config.use_cache = False #Desativa qualquer conflito com KV Cache durante o treino


In [ ]:
dataset = load_dataset ("parquet", data_files = "FugaziP75.parquet", split = "train") #https://huggingface.co/docs/datasets/loading

<hr>

In [ ]:
data_collator = DataCollatorForLanguageModeling (tokenizer = TOKENIZER, mlm = False)
#https://huggingface.co/docs/transformers/main_classes/data_collator
#mlm = False é muito importante para não aplicar Mask

#É preciso ter atenção aqui porque no meu caso, o meu dataset já vem tokenizado e com Padding = False mas pode não ser o caso mais comum.

<hr>

In [ ]:
#https://huggingface.co/docs/peft/package_reference/lora#peft.LoraConfig
LoRA = LoraConfig (

    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"], #Treinar a MLP nem sempre traz muitas mais vantagens comparado com o tempo de treino.
    r = CONFIG["LoRA"]["rank"],
    lora_alpha = CONFIG["LoRA"]["alpha"], #Escala da influência do adapter
    lora_dropout = CONFIG["LoRA"]["dropout"], #Básico dos Transformers, para evitar overfitting
    init_lora_weights = True, #Não começa os pesos de maneira totalmente aleatória #Melhora a convergência
    bias = "none", #Não treinar os bias, reduz número de params
    task_type = "CAUSAL_LM" #Tipo de Task, neste caso é para LM

)
#https://huggingface.co/docs/peft/package_reference/lora#peft.LoraConfig.loftq_config #https://arxiv.org/pdf/2310.08659
#LoftQ podia ser interessante porém, não é possível dar load do model diretamente na GPU. Para usar LoftQ o modelo não pode chegar quantizado.
#---------------------------------------------------

#Análise de LoRA
MODELO = get_peft_model (MODELO, LoRA) #get_peft_model aplica LoRA ao modelo. Aqui é apenas utilizado para análise do número de parâmetros
MODELO.print_trainable_parameters()

#trainable params: 15,335,424 || all params: 8,206,070,784 || trainable%: 0.1869


In [ ]:

Args_Treino = TrainingArguments (

    learning_rate = 2e-4,
    per_device_train_batch_size = CONFIG["Training_Args"]["micro_batch"],
    gradient_accumulation_steps = CONFIG["Training_Args"]["effective_batch"],
    logging_steps = 5, #logs de x em x steps
    fp16 = True,
    max_steps = CONFIG["Training_Args"]["max_steps"], #max_steps = gradient_accumulation_steps * per_device_train_batch_size / número de exemplos no dataset
    optim = "adamw_8bit", #IMPORTANTE
    lr_scheduler_type = "cosine", #Scheduler Learning Rate
    warmup_steps = CONFIG["Training_Args"]["warmup_steps"]
)

In [ ]:
class MonitorTraining (TrainerCallback):

  def __init__ (self):

    nvmlInit ()
    self.GPU = nvmlDeviceGetHandleByIndex (0)

    self.POWER = []

    self.MEM_GPU = []
    self.TORCH_MEM_GPU_ALLOC = []
    self.TORCH_MEM_GPU_RESERV = []
    self.UTIL_RATE_MEM_GPU = []
    self.UTIL_RATE_GPU = []
    self.GPU_MEM_CLOCK = []
    self.GPU_SM_CLOCK = []
    self.TEMPERATURA_GPU = []

    self.CPU_MEM = []
    self.CPU_UTIL = []

    self.TEMPO_STEP = []
    self.TOKENS_s = []

  def on_train_begin (self, args, state, control, **kwargs):

    torch.cuda.reset_peak_memory_stats () #Resetar dados de memória para não haver dados falsos
    torch.cuda.empty_cache () #Limpar cache para o treino começar limpo

    logging.info ("Training Arguments: %s", args.to_dict ())

    torch.cuda.synchronize ()
    self.TRAINING_TIME_START = time.time ()

    self.POWER.append (nvmlDeviceGetPowerUsage (self.GPU) / 1000)

  def on_step_begin (self, args, state, control, **kwargs):

    logging.info ("Optimizer: %s", state.global_step)

    torch.cuda.synchronize ()
    self.STEP_TIME_START = time.time ()

    self.POWER.append (nvmlDeviceGetPowerUsage (self.GPU) / 1000)

  def on_substep_end (self, args, state, control, **kwargs):

    ### Potência
    self.POWER.append (nvmlDeviceGetPowerUsage (self.GPU) / 1000) #Retorna em mW, /1000 = W

    ### GPU
    MEM = nvmlDeviceGetMemoryInfo (self.GPU)

    self.MEM_GPU.append (MEM.used / 1024**3) # GB
    self.TORCH_MEM_GPU_ALLOC.append (torch.cuda.memory_allocated () / 1024**3) # GB
    self.TORCH_MEM_GPU_RESERV.append (torch.cuda.memory_reserved () / 1024**3) # GB
    self.UTIL_RATE_MEM_GPU.append (MEM.used / MEM.total * 100) # %

    self.UTIL_RATE_GPU.append (nvmlDeviceGetUtilizationRates (self.GPU).gpu) # %

    self.GPU_MEM_CLOCK.append (nvmlDeviceGetClockInfo (self.GPU, NVML_CLOCK_MEM)) # Hz
    self.GPU_SM_CLOCK.append (nvmlDeviceGetClockInfo (self.GPU, NVML_CLOCK_SM)) # Hz

    self.TEMPERATURA_GPU.append (nvmlDeviceGetTemperature (self.GPU, NVML_TEMPERATURE_GPU)) # Celsius

    ### CPU
    self.CPU_MEM.append (psutil.virtual_memory().used / 1024**3) # GB
    self.CPU_UTIL.append (psutil.cpu_percent (interval = None)) # %

  def on_step_end (self, args, state, control, **kwargs):

    ### GPU
    GPU_MEM_USADA_TORCH = torch.cuda.memory_allocated() / 1024**3
    GPU_MEM_RESERVADA_TORCH = torch.cuda.memory_reserved() / 1024**3
    GPU_MEM_USADA = nvmlDeviceGetMemoryInfo (self.GPU).used / 1024**3

    ### CPU
    CPU_MEM = psutil.virtual_memory ()

    ### Performance
    torch.cuda.synchronize()
    TEMPO_STEP_LOG = time.time () - self.STEP_TIME_START
    self.TEMPO_STEP.append (TEMPO_STEP_LOG)
    TOKENS_s_LOG = CONFIG["Dataset"]["sequence_len"] * args.gradient_accumulation_steps / TEMPO_STEP_LOG
    self.TOKENS_s.append (TOKENS_s_LOG)

    logging.info ("Memória GPU Effective Usada: %.2f GB", GPU_MEM_USADA )
    logging.info ("Memória Torch GPU Usada: %.2f GB", GPU_MEM_USADA_TORCH)
    logging.info ("Memória Torch GPU Reservada: %.2f GB", GPU_MEM_RESERVADA_TORCH)
    logging.info ("Memória CPU Usada: %.2f GB", CPU_MEM.used / 1024**3)
    logging.info ("Tempo do Step: %s s", TEMPO_STEP_LOG)
    logging.info ("Tokens/s: %s tok/s", TOKENS_s_LOG)

    self.POWER.append (nvmlDeviceGetPowerUsage (self.GPU) / 1000)

  def on_train_end (self, args, state, control, **kwargs):

    """
    Este método vai ser dividio em duas fases:
    1ª - Estatísticas do Treino Realizado
    2ª - Estatísticas do Treino Real

    Este estudo está a ser realizado com exemplos com max len do verdadeiro dataset para calcular métricas para o treino real.
    """

    logging.info ("Estatísticas do Treino Realizado")

    for data in state.log_history:
      logging.info ("%s", data)

    logging.info ("Estatísticas do Treino Realizado")

    logging.info ("Logs Memória GPU: %s", self.MEM_GPU)
    logging.info ("Max Memória GPU: %s", np.max(self.MEM_GPU))
    logging.info ("Média Memória GPU: %s", np.mean(self.MEM_GPU))
    logging.info ("P95 Memória GPU: %s", np.percentile(self.MEM_GPU, 95))
    logging.info ("Desvio Padrão Memória GPU: %s", np.std(self.MEM_GPU))

    logging.info ("Logs Memória Alocada GPU Torch: %s", self.TORCH_MEM_GPU_ALLOC)
    logging.info ("Max Memória Alocada GPU Torch: %s", np.max(self.TORCH_MEM_GPU_ALLOC))
    logging.info ("Média Memória Alocada GPU Torch: %s", np.mean(self.TORCH_MEM_GPU_ALLOC))
    logging.info ("P95 Memória Alocada GPU Torch: %s", np.percentile(self.TORCH_MEM_GPU_ALLOC, 95))
    logging.info ("Desvio Padrão Memória Alocada GPU Torch: %s", np.std(self.TORCH_MEM_GPU_ALLOC))

    logging.info ("Logs Memória Reservada GPU Torch: %s", self.TORCH_MEM_GPU_RESERV)
    logging.info ("Max Memória Reservada GPU Torch: %s", np.max(self.TORCH_MEM_GPU_RESERV))
    logging.info ("Média Memória Reservada GPU Torch: %s", np.mean(self.TORCH_MEM_GPU_RESERV))
    logging.info ("P95 Memória Reservada GPU Torch: %s", np.percentile(self.TORCH_MEM_GPU_RESERV, 95))
    logging.info ("Desvio Padrão Memória Reservada GPU Torch: %s", np.std(self.TORCH_MEM_GPU_RESERV))

    logging.info ("Logs Percentagem de Utilização da Memória GPU: %s", self.UTIL_RATE_MEM_GPU)
    logging.info ("Max Percentagem de Utilização da Memória GPU: %s", np.max(self.UTIL_RATE_MEM_GPU))
    logging.info ("Média Percentagem de Utilização da Memória GPU: %s", np.mean(self.UTIL_RATE_MEM_GPU))
    logging.info ("P95 Percentagem de Utilização da Memória GPU: %s", np.percentile(self.UTIL_RATE_MEM_GPU, 95))
    logging.info ("Desvio Padrão Percentagem de Utilização da Memória GPU: %s", np.std(self.UTIL_RATE_MEM_GPU))

    logging.info ("Logs Percentagem de Utilização da GPU: %s", self.UTIL_RATE_GPU)
    logging.info ("Max Percentagem de Utilização da GPU: %s", np.max(self.UTIL_RATE_GPU))
    logging.info ("Média Percentagem de Utilização da GPU: %s", np.mean(self.UTIL_RATE_GPU))
    logging.info ("P95 Percentagem de Utilização da GPU: %s", np.percentile(self.UTIL_RATE_GPU, 95))
    logging.info ("Desvio Padrão Percentagem de Utilização da GPU: %s", np.std(self.UTIL_RATE_GPU))

    logging.info ("Logs Clock Memória GPU: %s", self.GPU_MEM_CLOCK)
    logging.info ("Max Clock Memória GPU: %s", np.max(self.GPU_MEM_CLOCK))
    logging.info ("Média Clock Memória GPU: %s", np.mean(self.GPU_MEM_CLOCK))
    logging.info ("P95 Clock Memória GPU: %s", np.percentile(self.GPU_MEM_CLOCK, 95))
    logging.info ("Desvio Padrão Clock Memória GPU: %s", np.std(self.GPU_MEM_CLOCK))

    logging.info ("Logs Clock SM GPU: %s", self.GPU_SM_CLOCK)
    logging.info ("Max Clock SM GPU: %s", np.max(self.GPU_SM_CLOCK))
    logging.info ("Média Clock SM GPU: %s", np.mean(self.GPU_SM_CLOCK))
    logging.info ("P95 Clock SM GPU: %s", np.percentile(self.GPU_SM_CLOCK, 95))
    logging.info ("Desvio Padrão Clock SM GPU: %s", np.std(self.GPU_SM_CLOCK))

    logging.info ("Logs Temperatura GPU: %s", self.TEMPERATURA_GPU)
    logging.info ("Max Temperatura GPU: %s", np.max(self.TEMPERATURA_GPU))
    logging.info ("Média Temperatura GPU: %s", np.mean(self.TEMPERATURA_GPU))
    logging.info ("P95 Temperatura GPU: %s", np.percentile(self.TEMPERATURA_GPU, 95))
    logging.info ("Desvio Padrão Temperatura GPU: %s", np.std(self.TEMPERATURA_GPU))

    #####

    logging.info ("Logs Memória CPU: %s", self.CPU_MEM)
    logging.info ("Max Memória CPU: %s", np.max(self.CPU_MEM))
    logging.info ("Média Memória CPU: %s", np.mean(self.CPU_MEM))
    logging.info ("P95 Memória CPU: %s", np.percentile(self.CPU_MEM, 95))
    logging.info ("Desvio Padrão Memória CPU: %s", np.std(self.CPU_MEM))

    logging.info ("Logs Utilização CPU: %s", self.CPU_UTIL)
    logging.info ("Max Utilização CPU: %s", np.max(self.CPU_UTIL))
    logging.info ("Média Utilização CPU: %s", np.mean(self.CPU_UTIL))
    logging.info ("P95 Utilização CPU: %s", np.percentile(self.CPU_UTIL, 95))
    logging.info ("Desvio Padrão Utilização CPU: %s", np.std(self.CPU_UTIL))

    #####

    logging.info ("Logs Tokens/s: %s", self.TOKENS_s)
    logging.info ("Max Tokens/s: %s", np.max(self.TOKENS_s))
    logging.info ("Média Tokens/s: %s", np.mean(self.TOKENS_s))
    logging.info ("P95 Tokens/s: %s", np.percentile(self.TOKENS_s, 95))
    logging.info ("Desvio Padrão Tokens/s: %s", np.std(self.TOKENS_s))

    logging.info ("Logs Tempo Step: %s", self.TEMPO_STEP)
    logging.info ("Max Tempo Step: %s", np.max(self.TEMPO_STEP))
    logging.info ("Média Tempo Step: %s", np.mean(self.TEMPO_STEP))
    logging.info ("P95 Tempo Step: %s", np.percentile(self.TEMPO_STEP, 95))
    logging.info ("Desvio Padrão Tempo Step: %s", np.std(self.TEMPO_STEP))

    TIME_TRAINING_TOTAL = time.time () - self.TRAINING_TIME_START
    logging.info ("Tempo Total de Treino: %s", TIME_TRAINING_TOTAL)
    logging.info ("Tempo por Effective Batch (Step): %s", TIME_TRAINING_TOTAL / state.max_steps)
    logging.info ("Tempo por Micro Batch: %s", TIME_TRAINING_TOTAL / state.max_steps / args.gradient_accumulation_steps)
    logging.info ("Tokens/s: %s", CONFIG["Dataset"]["sequence_len"] * args.gradient_accumulation_steps * state.max_steps / TIME_TRAINING_TOTAL) #Treino Total

    logging.info ("Logs Potência: %s", self.POWER)
    logging.info ("Max Potência: %s", np.max(self.POWER))
    logging.info ("Média Potência: %s", np.mean(self.POWER))
    logging.info ("P95 Potência: %s", np.percentile(self.POWER, 95))
    logging.info ("Desvio Padrão Potência: %s", np.std(self.POWER))

    ENERGIA = (np.percentile(self.POWER, 75) * TIME_TRAINING_TOTAL) / 3600 # Como o tempo está em segundos, sem / 3600 tens em Joules mas é melhor por Watt Hora
    ENERGIA_EFFECTIVE_BATCH = ENERGIA / state.max_steps
    ENERGIA_MICRO_BATCH = ENERGIA / (args.per_device_train_batch_size * state.max_steps * args.gradient_accumulation_steps)

    ENERGIA_TOKEN = ENERGIA / (CONFIG["Dataset"]["sequence_len"] * args.per_device_train_batch_size * args.gradient_accumulation_steps * state.max_steps) # Mais preciso
    ENERGIA_1MILION_TOKENS = ENERGIA_TOKEN * 1_000_000
    ENERGIA_10MILION_TOKENS = ENERGIA_TOKEN * 10_000_000
    ENERGIA_100MILION_TOKENS = ENERGIA_TOKEN * 100_000_000

    logging.info ("Energia Consumida: %s Wh", ENERGIA)
    logging.info ("Energia por Effective Batch (Step): %s Wh", ENERGIA_EFFECTIVE_BATCH)
    logging.info ("Energia por Micro Batch: %s Wh", ENERGIA_MICRO_BATCH)
    logging.info ("Energia por Token: %s Wh", ENERGIA_TOKEN)
    logging.info ("Energia por 1 Milhão de Tokens: %s Wh", ENERGIA_1MILION_TOKENS)
    logging.info ("Energia por 10 Milhões de Tokens: %s Wh", ENERGIA_10MILION_TOKENS)
    logging.info ("Energia por 100 Milhões de Tokens: %s Wh", ENERGIA_100MILION_TOKENS)


In [ ]:
trainer = Trainer ( #https://huggingface.co/docs/transformers/main_classes/trainer
    model = MODELO,
    args = Args_Treino,
    train_dataset = dataset,
    data_collator = data_collator,
    callbacks = [MonitorTraining()],
)

#packing ???

In [ ]:
trainer.train()